In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from skimage.feature import hog
from skimage.color import rgb2gray
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
IMAGE_DATA_DIR = Path("asd_images")
TEXT_CSV_PATH = Path("asd_text.csv")
IMAGE_SIZE = 128

# =============================
# 1. DATASET LOADING
# =============================

def load_image_paths(root: Path) -> List[Tuple[str, int]]:
    pairs = []
    train_dir = root / "Train"
    
    # আপনার ফোল্ডারের নামগুলো এখানে পরিষ্কারভাবে উল্লেখ করুন
    # যদি ফোল্ডারের নাম 'autism' হয় তবে লেবেল ১, 'tipical' হলে ০
    mapping = {"autism": 1, "autistic": 1, "tipical": 0}
    
    for folder_name in ["autism", "autistic", "tipical"]:
        class_dir = train_dir / folder_name
        if class_dir.exists():
            # ইমেজ সংখ্যা চেক করা
            images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
            print(f"Checking {class_dir}: Found {len(images)} images.")
            
            for p in images:
                pairs.append((str(p), mapping[folder_name]))
        else:
            print(f"Folder not found: {class_dir}")
            
    return pairs

def extract_hog_features(path: str) -> np.ndarray:
    img = Image.open(path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
    gray = rgb2gray(np.asarray(img))
    return hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), transform_sqrt=True, feature_vector=True)

# =============================
# 2. IMAGE MODEL
# =============================

image_pairs = load_image_paths(IMAGE_DATA_DIR)
X_img = np.vstack([extract_hog_features(p) for p, _ in image_pairs])
y_img = np.array([lbl for _, lbl in image_pairs])

scaler_img = StandardScaler()
X_img_scaled = scaler_img.fit_transform(X_img)
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_img_scaled, y_img, test_size=0.2, random_state=RANDOM_STATE)

image_clf = SGDClassifier(loss="log_loss", max_iter=1000, random_state=RANDOM_STATE)
image_clf.fit(X_train_img, y_train_img)

# =============================
# 3. TEXT MODEL
# =============================

text_df = pd.read_csv(TEXT_CSV_PATH)
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(text_df["note"], text_df["label"], test_size=0.2, random_state=RANDOM_STATE)

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("clf", LogisticRegression(max_iter=300)),
])
text_pipeline.fit(X_train_text, y_train_text)

# =============================
# 4. FUSION & PREDICTION
# =============================

def predict_from_user_input(image_path, text_note, method="weighted"):
    probas = {}
    if image_path:
        feat = extract_hog_features(image_path).reshape(1, -1)
        probas["image"] = float(image_clf.predict_proba(scaler_img.transform(feat))[0, 1])
    if text_note:
        probas["text"] = float(text_pipeline.predict_proba([text_note])[0, 1])
    
    # Weighted Fusion
    if method == "weighted":
        score = (probas.get("image", 0.5) * 0.65) + (probas.get("text", 0.5) * 0.35)
    else:
        score = np.mean(list(probas.values()))
        
    label = "ASD" if score >= 0.5 else "No ASD"
    return {"score": score, "label": label, "probas": probas}

# =============================
# 5. EXECUTION
# =============================

user_img = input("Enter image path: ")
user_txt = input("Enter note: ")
result = predict_from_user_input(user_img, user_txt)

print(f"Prediction: {result['label']} (Score: {result['score']:.4f})")

Checking asd_images\Train\autism: Found 1268 images.
Folder not found: asd_images\Train\autistic
Checking asd_images\Train\tipical: Found 1268 images.
